In [1]:
import pandas as pd
import os
import json

In [2]:
def finetune_results(root_path):
    df = pd.DataFrame({
        'dataset': [], 'model': [], 
        'loss': [], 'object_acc': [], 'best_region': [], 'worst_region': [], 
        'best_region_acc': [], 'worst_region_acc': [], 'region_bias_gap': [], 'finetune': [],
    })

    for dataset in os.listdir(root_path):
        if dataset != 'imagenet':
            dataset_path = os.path.join(root_path, dataset)
            for model in os.listdir(dataset_path):
                model_path = os.path.join(dataset_path, model)
                for file in os.listdir(model_path):
                    if "results" in file:
                        result_path = os.path.join(model_path, file)

                        with open(result_path, 'r') as f:
                            data_dict = json.load(f)
                            df.loc[len(df)] = [
                                dataset,
                                model,
                                data_dict['loss'],
                                data_dict['object_acc'],
                                data_dict['best_region'],
                                data_dict['worst_region'],
                                data_dict['best_region_acc'],
                                data_dict['worst_region_acc'],
                                data_dict['region_bias_gap'],
                                'yes'
                            ]
        else:
            dataset_path = os.path.join(root_path, dataset)
            for file in os.listdir(dataset_path):
                result_path = os.path.join(dataset_path, file)
                dataset = file.split('_')[0]
                model = file.split('_')[1][:-5]
                with open(result_path, 'r') as f:
                    data_dict = json.load(f)['splits']['test']
                    df.loc[len(df)] = [
                        dataset,
                        model,
                        data_dict['loss'],
                        data_dict['object_acc'],
                        data_dict['best_region'],
                        data_dict['worst_region'],
                        data_dict['best_region_acc'],
                        data_dict['worst_region_acc'],
                        data_dict['region_bias_gap'],
                        'no'
                    ]

    return df

In [3]:
root_path = r"D:\Project\DAP_paper\results"
results = finetune_results(root_path)

model_map = {
    "clip": "CLIP",
    "convnext": "ConvNeXt V2",
    "deit": "DeiT III",
    "dino": "DINOv2",
    "mae": "MAE",
    "maxvit": "MaxViT",
    "resnet": "ResNet-50",
    "siglip": "SigLIP",
    "swin": "Swin V2"
}

region_map = {
    # DollarStreet region codes
    "af": "Africa",
    "eu": "Europe",
    "am": "Americas",
    "as": "Asia",

    # GeoDE region names
    "Africa": "Africa",
    "Europe": "Europe",
    "Americas": "Americas",
    "EastAsia": "East Asia",
    "WestAsia": "West Asia",
    "SouthEastAsia": "Southeast Asia"
}

results["model"] = results["model"].map(model_map)

for col in ["best_region", "worst_region"]:
    results[col] = results[col].replace(region_map)

metrics = ["object_acc", "best_region_acc", "worst_region_acc", "region_bias_gap"]

results = results.copy()
results[metrics] = round(results[metrics] * 100, 2)

In [4]:
def filter_df(df, dataset=None, finetuned=False):
    finetune_value = "yes" if finetuned else "no"

    filtered = df.copy()

    if dataset is not None:
        filtered = filtered[filtered["dataset"] == dataset]

    filtered = filtered[filtered["finetune"] == finetune_value]

    filtered = filtered.reset_index(drop=True)

    filtered = filtered.drop(
        columns=["dataset", "finetune", "loss"],
        errors="ignore"
    )

    return filtered

In [5]:
geode_finetuned = filter_df(results, dataset="geode", finetuned=True)
geode_pretrained = filter_df(results, dataset="geode", finetuned=False)

dollar_finetuned = filter_df(results, dataset="dollarstreet", finetuned=True)
dollar_pretrained = filter_df(results, dataset="dollarstreet", finetuned=False)

In [6]:
geode_finetuned

,model,object_acc,best_region,worst_region,best_region_acc,worst_region_acc,region_bias_gap
0,CLIP,95.38,West Asia,Southeast Asia,96.83,93.77,3.06
1,ConvNeXt V2,96.61,West Asia,Southeast Asia,97.88,95.59,2.29
2,DeiT III,96.28,West Asia,Southeast Asia,97.20,95.20,2.00
3,DINOv2,95.95,West Asia,Southeast Asia,97.14,94.77,2.38
4,MAE,92.86,West Asia,Southeast Asia,93.92,92.05,1.87
5,MaxViT,95.65,West Asia,Southeast Asia,96.62,94.70,1.92
6,ResNet-50,90.86,Europe,Southeast Asia,92.38,89.72,2.66
7,SigLIP,95.91,West Asia,Southeast Asia,97.30,94.52,2.78
8,Swin V2,96.25,West Asia,Southeast Asia,97.41,95.31,2.10


In [7]:
geode_pretrained

,model,object_acc,best_region,worst_region,best_region_acc,worst_region_acc,region_bias_gap
0,CLIP,4.51,East Asia,West Asia,4.98,3.33,1.65
1,ConvNeXt V2,2.39,Americas,East Asia,2.87,1.80,1.07
2,DeiT III,1.77,Southeast Asia,Americas,1.93,1.54,0.39
3,DINOv2,2.10,Southeast Asia,West Asia,2.62,1.53,1.08
4,MAE,3.29,Southeast Asia,Africa,4.37,2.57,1.80
5,MaxViT,2.53,West Asia,East Asia,3.12,1.93,1.19
6,ResNet-50,2.82,Americas,Africa,3.51,2.13,1.38
7,SigLIP,2.52,West Asia,Europe,3.12,1.99,1.13
8,Swin V2,2.03,Europe,Africa,2.38,1.71,0.67


In [8]:
dollar_finetuned

,model,object_acc,best_region,worst_region,best_region_acc,worst_region_acc,region_bias_gap
0,CLIP,73.05,Europe,Africa,77.84,67.77,10.07
1,ConvNeXt V2,76.67,Europe,Africa,80.61,72.14,8.48
2,DeiT III,75.12,Europe,Africa,78.57,69.66,8.91
3,DINOv2,73.40,Europe,Africa,76.68,68.95,7.73
4,MAE,66.74,Europe,Africa,69.10,62.22,6.88
5,MaxViT,72.86,Europe,Africa,77.26,66.23,11.03
6,ResNet-50,58.87,Europe,Africa,63.56,51.36,12.20
7,SigLIP,76.21,Europe,Africa,79.45,72.02,7.43
8,Swin V2,76.09,Europe,Africa,79.15,71.90,7.25


In [9]:
dollar_pretrained

,model,object_acc,best_region,worst_region,best_region_acc,worst_region_acc,region_bias_gap
0,CLIP,1.46,Europe,Africa,1.75,1.18,0.57
1,ConvNeXt V2,1.60,Americas,Africa,2.34,1.06,1.27
2,DeiT III,1.90,Asia,Americas,2.08,1.52,0.57
3,DINOv2,1.07,Africa,Europe,1.53,0.44,1.10
4,MAE,2.60,Americas,Europe,2.92,2.19,0.73
5,MaxViT,2.88,Europe,Asia,4.52,1.98,2.54
6,ResNet-50,2.25,Europe,Americas,2.62,1.99,0.64
7,SigLIP,1.65,Asia,Europe,1.82,1.31,0.51
8,Swin V2,1.86,Africa,Asia,2.24,1.62,0.63


In [10]:
results[results['model'].isin(['ResNet-50', 'ConvNeXt V2'])]

,dataset,model,loss,object_acc,best_region,worst_region,best_region_acc,worst_region_acc,region_bias_gap,finetune
1,dollarstreet,ConvNeXt V2,1.427907,76.67,Europe,Africa,80.61,72.14,8.48,yes
6,dollarstreet,ResNet-50,2.005920,58.87,Europe,Africa,63.56,51.36,12.20,yes
10,geode,ConvNeXt V2,0.782145,96.61,West Asia,Southeast Asia,97.88,95.59,2.29,yes
15,geode,ResNet-50,0.953754,90.86,Europe,Southeast Asia,92.38,89.72,2.66,yes
19,dollarstreet,ConvNeXt V2,4.286775,1.60,Americas,Africa,2.34,1.06,1.27,no
24,dollarstreet,ResNet-50,4.165418,2.25,Europe,Americas,2.62,1.99,0.64,no
28,geode,ConvNeXt V2,3.804696,2.39,Americas,East Asia,2.87,1.80,1.07,no
33,geode,ResNet-50,3.695092,2.82,Americas,Africa,3.51,2.13,1.38,no
